In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import torch

# If there's a GPU available...
if torch.cuda.is_available():

    # Tell PyTorch to use the GPU.
    device = torch.device("cuda:1")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))
    !nvidia-smi

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

There are 4 GPU(s) available.
We will use the GPU: NVIDIA A40
Tue Dec 31 07:34:40 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.42.06              Driver Version: 555.42.06      CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     Off |   00000000:01:00.0 Off |                    0 |
|  0%   65C    P0            278W /  300W |   34887MiB /  46068MiB |     99%      Default |
|                                         |                        |          

In [4]:
from transformers import LongformerTokenizerFast, LongformerForSequenceClassification, Trainer, TrainingArguments, LongformerConfig
from transformers import LongformerTokenizer

/home/abdoah/anaconda3/envs/m24/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from data import ECHRDataset

In [6]:
train_files_path = '/home/abdoah/TND/legal-bert/train-files.txt'
eval_files_path = '/home/abdoah/TND/legal-bert/eval-files.txt'
data_path = '/home/abdoah/data/echr-processed'

tokenizer_name = 'allenai/longformer-base-4096'
model_name = 'allenai/longformer-base-4096'

max_length = 4096

tokenizer = LongformerTokenizer.from_pretrained(model_name)

train_dataset = ECHRDataset(train_files_path, data_path, tokenizer, max_length)
eval_dataset = ECHRDataset(train_files_path, data_path, tokenizer, max_length)

Using Tokenizer allenai/longformer-base-4096
Using Tokenizer allenai/longformer-base-4096


In [7]:
item = train_dataset[0]
print(item['input_ids'].shape), print(item['attention_mask'].shape), print(item['label'].shape)

Label: 2
tensor(2)
torch.Size([4096])
torch.Size([4096])
torch.Size([])


(None, None, None)

In [8]:
input_ids = item['input_ids']
input_ids = input_ids.squeeze(0)
print(input_ids.shape)

torch.Size([4096])


In [9]:
from transformers import LongformerModel
from transformers import DataCollatorWithPadding, LongformerForSequenceClassification

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
model = LongformerForSequenceClassification.from_pretrained(model_name, num_labels=3, gradient_checkpointing=True)

Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at allenai/longformer-base-4096 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
import numpy as np
from sklearn.metrics import f1_score
 
# Metric helper method
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    score = f1_score(
            labels, predictions, labels=labels, pos_label=1, average="macro"
        )
    return {"f1": float(score) if score == 1 else score}


In [11]:
def set_seed(seed=42):
  import random
  import numpy as np
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.deterministic=True
  torch.backends.cudnn.benchmark = False

In [12]:

from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_longformer_3_classes",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
    do_eval = True,
    evaluation_strategy = 'epoch',
    save_strategy = 'epoch',
    load_best_model_at_end = True, # this allows to automatically get the best model at the end based on whatever metric we want
    metric_for_best_model = 'macro_f1',
    greater_is_better = True,
    report_to='none',  # Disable wandb integration
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

set_seed(training_args.seed)
trainer.train()



/home/abdoah/anaconda3/envs/m24/lib/python3.12/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_2040189/2884453500.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Label: 1
tensor(1)
Label: 1
tensor(1)
Label: 1
tensor(1)
Label: 2
tensor(2)
Label: 2
tensor(2)
Label: 1
tensor(1)
Label: 2
tensor(2)
Label: 1
tensor(1)


Initializing global attention on CLS token...
Initializing global attention on CLS token...
Initializing global attention on CLS token...


: 

: 